In [ ]:
import os
import time
import certifi
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from requests.packages.urllib3.exceptions import InsecureRequestWarning


In [ ]:
base_url = catalog.load('params:oai_extract_options.base_url')
context = catalog.load('params:oai_extract_options.context')

env = 'dev'

print("base_url: ", base_url)
print("context: ", context)

In [ ]:
def get_oai_response(base_url, verify=None, max_retries=3, backoff_factor=1.0, min_interval=0.0):

    # Usa el bundle de certifi para evitar errores de certificado en requests
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    VERIFY_SSL = os.getenv("OAI_VERIFY_SSL", "false").lower() == "true"
    CA_BUNDLE = os.getenv("OAI_CA_BUNDLE") or certifi.where()
    requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)

    verify_param = CA_BUNDLE if VERIFY_SSL else False
    if verify is not None:
        verify_param = verify

    for attempt in range(1, max_retries + 1):
        start_time = time.time()
        response = None
        error = None
        try:
            response = requests.get(base_url, verify=verify_param)
        except requests.RequestException as exc:
            error = exc
        elapsed_time = time.time() - start_time

        if min_interval > 0:
            wait_time = max(min_interval - elapsed_time, 0)
            if wait_time > 0:
                print(f"Pausando {wait_time:.2f} segundos para no saturar el servidor")
                time.sleep(wait_time)

        if error:
            print(f"Error en request (intento {attempt}/{max_retries}): {error}")

        if response and response.status_code == 200:
            return response

        status = response.status_code if response else "sin respuesta"
        print(f"Error: {status} (intento {attempt}/{max_retries})")

        if attempt < max_retries:
            backoff = backoff_factor * attempt
            print(f"Reintentando en {backoff:.2f} segundos...")
            time.sleep(backoff)
    return None

def log_oai_progress(token_elem, total_processed: int):
    """Muestra el avance usando completeListSize y los registros acumulados."""
    if token_elem is None:
        return
    total = token_elem.get('completeListSize')
    try:
        total_int = int(total) if total is not None else None
        if total_int is not None and total_processed is not None:
            remaining = total_int - total_processed
            print(f"Progreso OAI: {total_processed}/{total_int} (faltan ~{remaining})")
    except ValueError:
        # Si el servidor devuelve valores no numéricos, ignora el progreso.
        pass


## Extract identifiers 

In [ ]:
def oai_extract_records(base_url: str, context: str, env: str, verify=None) -> pd.DataFrame:
    records = []
    
    iteration_limit = 2 if env == "dev" else None
    resumption_token = None
    iteration_count = 0

    total_processed = 0

    while True:
        if iteration_limit is not None and iteration_count >= iteration_limit:
            break

        if resumption_token:
            params = f'/{context}?verb=ListRecords&resumptionToken={resumption_token}'
        else:
            params = f'/{context}?verb=ListRecords&metadataPrefix=oai_dc'

        url = base_url + params

        print(f"Consultando: {url}")

        response = get_oai_response(url, verify=verify)

        iteration_count += 1

        if not response or not response.ok:
            print(f"Error al consultar: {url}")
            break

        xml_content = response.text
        root = ET.fromstring(xml_content)
        ns = {
            'oai': 'http://www.openarchives.org/OAI/2.0/',
            'dc': 'http://purl.org/dc/elements/1.1/'
        }

        record_nodes = root.findall('.//oai:record', ns)

        if not record_nodes:
            print("No se encontraron más registros.")
            break

        for record in record_nodes:
            header = record.find('.//oai:header', ns)
            identifier_node = header.find('.//oai:identifier', ns) if header is not None else None
            datestamp_node = header.find('.//oai:datestamp', ns) if header is not None else None
            setspec = [e.text for e in header.findall('.//oai:setSpec', ns)] if header is not None else []

            metadata = record.find('.//oai:metadata', ns)

            if metadata is None:
                continue

            # Valores simples
            title = metadata.find('.//dc:title', ns)
            date_issued = metadata.find('.//dc:date', ns)

            # Multivaluados
            setspec = [e.text for e in record.findall('.//oai:setSpec', ns)]

            creators = [e.text for e in metadata.findall('.//dc:creator', ns)]
            description = [e.text for e in metadata.findall('.//dc:description', ns)]
            types = [e.text for e in metadata.findall('.//dc:type', ns)]
            identifiers = [e.text for e in metadata.findall('.//dc:identifier', ns)]
            languages = [e.text for e in metadata.findall('.//dc:language', ns)]
            publishers = [e.text for e in metadata.findall('.//dc:publisher', ns)]
            subjects = [e.text for e in metadata.findall('.//dc:subject', ns)]
            relations = [e.text for e in metadata.findall('.//dc:relation', ns)]
            rights = [e.text for e in metadata.findall('.//dc:rights', ns)]
            formats = [e.text for e in metadata.findall('.//dc:format', ns)]

            records.append({
                'record_id': identifier_node.text if identifier_node is not None else None,
                'datestamp': datestamp_node.text if datestamp_node is not None else None,
                'set_id': setspec,
                'col_id': setspec[0] if setspec else None,
                'title': title.text if title is not None else None,
                'date_issued': date_issued.text if date_issued is not None else None,
                'creators': creators,
                'description': description,
                'types': types,
                'identifiers': identifiers,
                'languages': languages,
                'subjects': subjects,
                'publishers': publishers,
                'relations': relations,
                'rights': rights,
                'formats': formats
            })

        total_processed += len(record_nodes)

        token_elem = root.find('.//oai:resumptionToken', ns)
        resumption_token = token_elem.text if token_elem is not None else None
        log_oai_progress(token_elem, total_processed)

        if not resumption_token:
            break

    df = pd.DataFrame(records)

    timestamp = pd.Timestamp.now(tz="UTC").normalize()
    df['extract_datetime'] = timestamp
    df['_context'] = context

    return df, df.head(100)



In [ ]:
df_records, df_dev = oai_extract_records(base_url, context, env)

In [ ]:
df_records